In [1]:
!pip install nest_asyncio


In [2]:
import threading
import queue
import time
import random
import asyncio
from multiprocessing import Process
from datetime import datetime

In [3]:
class Paciente:
    contador_id = 0

    def __init__(self, nombre, edad, genero, sintomas, antecedentes=None):
        Paciente.contador_id += 1
        self.id = Paciente.contador_id
        self.nombre = nombre
        self.edad = edad
        self.genero = genero
        self.sintomas = sintomas
        self.antecedentes = antecedentes or []
        self.prioridad = "media"
        self.diagnostico = None

    def __str__(self):
        return f"{self.nombre} (ID:{self.id}, Edad:{self.edad}, Prioridad:{self.prioridad})"

def log_event(etapa, paciente):
    print(f"[{etapa}] ({datetime.now().strftime('%H:%M:%S')}) {paciente}")


In [4]:
cola_pacientes = queue.Queue()
computadoras = threading.BoundedSemaphore(3)

def registrar_paciente(paciente):
    with computadoras:
        log_event("Registro-Inicio", paciente)
        time.sleep(random.uniform(0.5, 1.0))
        cola_pacientes.put(paciente)
        log_event("Registro-Fin", paciente)

def iniciar_registro(pacientes):
    hilos = []
    for paciente in pacientes:
        hilo = threading.Thread(target=registrar_paciente, args=(paciente,))
        hilos.append(hilo)
        hilo.start()
    for hilo in hilos:
        hilo.join()


In [5]:
async def diagnosticar(paciente):
    log_event("Diagnóstico-Inicio", paciente)
    await asyncio.sleep(random.uniform(1.0, 2.0))
    diagnostico = random.choice(["leve", "moderado", "grave"])
    paciente.diagnostico = diagnostico
    if diagnostico == "grave":
        paciente.prioridad = "alta"
    elif diagnostico == "moderado":
        paciente.prioridad = "media"
    else:
        paciente.prioridad = "baja"
    log_event("Diagnóstico-Fin", paciente)
    return paciente

async def diagnosticar_pacientes(lista_pacientes):
    tareas = [diagnosticar(p) for p in lista_pacientes]
    resultados = await asyncio.gather(*tareas)
    return resultados


In [6]:
camas = threading.Semaphore(3)
doctores = threading.Semaphore(2)

def asignar_recursos(paciente):
    if paciente.prioridad == "baja":
        log_event("Derivado a consulta externa", paciente)
        return
    log_event("Espera de recursos", paciente)
    with camas, doctores:
        log_event("Atención-Inicio", paciente)
        time.sleep(random.uniform(1.0, 2.0))
        log_event("Atención-Fin", paciente)


In [7]:
def seguimiento(paciente):
    print(f"[Seguimiento] {paciente.nombre} en seguimiento intensivo...")
    time.sleep(random.uniform(1.0, 2.0))
    print(f"[Alta] {paciente.nombre} ha sido dado de alta.")

def iniciar_seguimiento(pacientes):
    procesos = []
    for paciente in pacientes:
        if paciente.prioridad in ["alta", "media"]:
            p = Process(target=seguimiento, args=(paciente,))
            procesos.append(p)
            p.start()
    for p in procesos:
        p.join()


In [9]:
pacientes = [
    Paciente("Ana", 30, "F", "Fiebre y tos"),
    Paciente("Luis", 40, "M", "Dolor en el pecho"),
    Paciente("María", 25, "F", "Dolor abdominal"),
    Paciente("Pedro", 50, "M", "Dificultad respiratoria"),
    Paciente("Valeria", 60, "F", "Dolor de espalda"),
    Paciente("José", 35, "M", "Mareos y náuseas"),
    Paciente("Lucía", 28, "F", "Dolor de garganta"),
]

print("\n INICIO DEL FLUJO\n")

iniciar_registro(pacientes)
registrados = list(cola_pacientes.queue)

import nest_asyncio
nest_asyncio.apply()
pacientes_diagnosticados = await diagnosticar_pacientes(registrados)

hilos = []
for paciente in pacientes_diagnosticados:
    hilo = threading.Thread(target=asignar_recursos, args=(paciente,))
    hilos.append(hilo)
    hilo.start()
for hilo in hilos:
    hilo.join()

iniciar_seguimiento(pacientes_diagnosticados)



 INICIO DEL FLUJO

[Registro-Inicio] (18:17:56) Ana (ID:8, Edad:30, Prioridad:media)
[Registro-Inicio] (18:17:56) Luis (ID:9, Edad:40, Prioridad:media)
[Registro-Inicio] (18:17:56) María (ID:10, Edad:25, Prioridad:media)
[Registro-Fin] (18:17:56) María (ID:10, Edad:25, Prioridad:media)
[Registro-Inicio] (18:17:56) Pedro (ID:11, Edad:50, Prioridad:media)
[Registro-Fin] (18:17:56) Luis (ID:9, Edad:40, Prioridad:media)
[Registro-Inicio] (18:17:56) Valeria (ID:12, Edad:60, Prioridad:media)
[Registro-Fin] (18:17:56) Ana (ID:8, Edad:30, Prioridad:media)
[Registro-Inicio] (18:17:56) José (ID:13, Edad:35, Prioridad:media)
[Registro-Fin] (18:17:57) Valeria (ID:12, Edad:60, Prioridad:media)
[Registro-Inicio] (18:17:57) Lucía (ID:14, Edad:28, Prioridad:media)
[Registro-Fin] (18:17:57) Pedro (ID:11, Edad:50, Prioridad:media)
[Registro-Fin] (18:17:57) José (ID:13, Edad:35, Prioridad:media)
[Registro-Fin] (18:17:58) Lucía (ID:14, Edad:28, Prioridad:media)
[Diagnóstico-Inicio] (18:17:58) Ana (ID:1, 